# D2 - Iterators, Generators & Memory-Efficient Data Handling

## Objective
Demonstrate memory-efficient data processing patterns in Python using custom iterators (`__iter__`, `__next__`), generator functions (`yield`), and streaming batch processors. Compare eager loading with lazy evaluation and measure peak memory consumption.

## Concepts Covered
- **Custom Iterators (`__iter__`, `__next__`)**: Implementing stateful batch iterators (`FileBatchIterator`).
- **Generator Functions (`yield`)**: Creating memory-efficient data pipelines (`read_files_in_batches`).
- **Lazy Evaluation vs Eager Loading**: Loading data on demand versus accumulating entire datasets in RAM (`load_all_files`).
- **Peak Memory Benchmarking**: Using Python's `tracemalloc` to measure RAM usage under different data processing strategies.

## Project Implementation
The file iterator utilities reside in `app/utils/file_iterator.py`:
- `FileBatchIterator`: Class implementing the iterator protocol to stream file contents in configurable batch sizes.
- `read_files_in_batches`: Generator function producing lists of file content batches using `yield`.
- `load_all_files`: Eager loading function that reads all matching files into a single list in memory.
- `scripts/memory_benchmark.py`: Benchmark script demonstrating peak RAM usage across synthetic datasets.

## Demonstration
Below, we demonstrate each iteration strategy on temporary files and execute the benchmark logic.

In [1]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import tempfile
from app.utils.file_iterator import FileBatchIterator, read_files_in_batches, load_all_files

# Create a temporary directory with sample text files
with tempfile.TemporaryDirectory() as temp_dir:
    dir_path = Path(temp_dir)
    for i in range(10):
        (dir_path / f"file_{i:02d}.txt").write_text(f"Line content for record {i}", encoding="utf-8")
    
    # 1. Eager Loading
    all_data = load_all_files(dir_path)
    print(f"Eager loaded {len(all_data)} files into RAM simultaneously.")

    # 2. FileBatchIterator (Lazy Iterator Class)
    iterator = FileBatchIterator(dir_path, batch_size=3)
    print("\nStreaming with FileBatchIterator (batch_size=3):")
    for batch_num, batch in enumerate(iterator, start=1):
        print(f"  Batch {batch_num}: {len(batch)} items loaded")

    # 3. read_files_in_batches (Generator Function)
    print("\nStreaming with read_files_in_batches generator (batch_size=4):")
    for batch_num, batch in enumerate(read_files_in_batches(dir_path, batch_size=4), start=1):
        print(f"  Batch {batch_num}: {len(batch)} items loaded")

Eager loaded 10 files into RAM simultaneously.

Streaming with FileBatchIterator (batch_size=3):
  Batch 1: 3 items loaded
  Batch 2: 3 items loaded
  Batch 3: 3 items loaded
  Batch 4: 1 items loaded

Streaming with read_files_in_batches generator (batch_size=4):
  Batch 1: 4 items loaded
  Batch 2: 4 items loaded
  Batch 3: 2 items loaded


In [2]:
import sys
from pathlib import Path

# Resolve project root directory regardless of execution context
root_dir = Path.cwd().resolve()
if root_dir.name == "notebooks":
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# Execute actual Memory Benchmark logic from scripts/memory_benchmark.py
from scripts.memory_benchmark import run_benchmark

results = run_benchmark()

2026-09-22 20:05:45 [INFO] app.scripts.memory_benchmark: ================================================================================


2026-09-22 20:05:45 [INFO] app.scripts.memory_benchmark:  D2 MEMORY BENCHMARK: EAGER vs LAZY ITERATOR vs GENERATOR


2026-09-22 20:05:45 [INFO] app.scripts.memory_benchmark:  Fixed Batch Size: 100 files | File Size: ~1 KB each


2026-09-22 20:05:45 [INFO] app.scripts.memory_benchmark: ================================================================================


2026-09-22 20:05:45 [INFO] app.scripts.memory_benchmark: Dataset Size    | Eager Peak (KB)    | Lazy Iterator (KB)   | Generator (KB)  


...

2026-09-22 20:06:06 [INFO] app.scripts.memory_benchmark: ================================================================================


2026-09-22 20:06:06 [INFO] app.scripts.memory_benchmark: Summary Observation:


2026-09-22 20:06:06 [INFO] app.scripts.memory_benchmark: - Eager loading memory scales linearly O(N) with the dataset size.


2026-09-22 20:06:06 [INFO] app.scripts.memory_benchmark: - Lazy Iterator & Generator memory remains approximately stable O(B) bounded by batch size.


2026-09-22 20:06:06 [INFO] app.scripts.memory_benchmark: ================================================================================


## Actual Output
The execution above yields:
1. Verification of eager loading ($O(N)$ memory scaling) vs batch streaming ($O(B)$ memory scaling).
2. Direct execution of `run_benchmark()` measuring memory usage in KB across dataset sizes of 100, 1,000, and 10,000 files.

## Key Observations
- Eager loading (`load_all_files`) loads all data into RAM at once, causing peak memory to scale linearly ($O(N)$) with the dataset size.
- Lazy Iterators (`FileBatchIterator`) and Generators (`read_files_in_batches`) consume memory proportional only to the batch size ($O(B)$), keeping peak memory constant regardless of whether processing 100 or 10,000 files.
- Generators eliminate the overhead of manually managing state variables (`self._index`) required by iterator classes.

## Conclusion
Using generators and custom iterators ensures the application scales efficiently to large datasets without exhausting available system RAM.